In [ ]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.agents.middleware import wrap_model_call
import json
from langchain.tools import tool
from pinecone import Pinecone
from langchain_pinecone import PineconeEmbeddings, PineconeVectorStore
from stockanalysis.yahoo_finance import get_latest_stock_price
from stockanalysis.utilities import (
    get_company_financials as fetch_company_financials,
    get_stock_history as fetch_stock_history,
    get_technical_indicators as fetch_technical_indicators,
)

Initialize OpenAI Model in Nebius, Pinecone Vector db

In [ ]:
load_dotenv()

llm = ChatOpenAI(
    model=os.environ["NEBIUS_MODEL"],
    api_key=os.environ["NEBIUS_API_KEY"],
    base_url="https://api.tokenfactory.nebius.com/v1/",
    temperature=0,
    use_responses_api=False,
)

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(os.environ["PINECONE_INDEX_NAME"])
print(f"Connected to Pinecone index: {index}")
embeddings = PineconeEmbeddings(
    model="llama-text-embed-v2",
    pinecone_api_key=os.environ["PINECONE_API_KEY"],
    query_params={
        "input_type": "query",
        "dimension": 512,
    },
    document_params={
        "input_type": "passage",
        "dimension": 512,
    },
)

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="stock-analysis-nsp",
)

print("Model:", llm.model_name)

Add required Tools

In [ ]:
from langchain.tools import tool

@tool
def get_latest_price(symbol: str) -> dict:
    """Get the latest available stock price bar for a ticker."""
    return get_latest_stock_price(symbol)


@tool
def get_company_financials(symbol: str, frequency: str = "quarterly") -> dict:
    """Get up to four income, balance-sheet, and cash-flow periods.

    frequency must be quarterly or annual. Values are reported historical
    financials, not forecasts.
    """
    return fetch_company_financials(symbol, frequency)


@tool
def get_stock_history(symbol: str, period: str = "3mo") -> dict:
    """Get adjusted daily OHLCV history for a stock.

    period must be 1mo, 3mo, 6mo, 1y, or 2y.
    """
    return fetch_stock_history(symbol, period)


@tool
def get_technical_indicators(symbol: str) -> dict:
    """Calculate 20/50/200-day SMAs and 14-day RSI from adjusted prices."""
    return fetch_technical_indicators(symbol)



@tool
def search_stock_lessons(query: str) -> str:
    """Search the PDF lessons for stock analysis and trading concepts."""
    documents = vector_store.similarity_search(query, k=4)

    print(f"[RAG] Retrieved {len(documents)} documents", flush=True)

    if not documents:
        return "No relevant passages found."

    return json.dumps(
        [
            {
                "content": doc.page_content,
                "source": doc.metadata.get("source"),
                "title": doc.metadata.get("title"),
                "page_number": doc.metadata.get("page_number"),
            }
            for doc in documents
        ],
        ensure_ascii=False,
    )

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

print("Creating agent...", flush=True)

agent = create_agent(
    model=llm,
    tools=[search_stock_lessons, get_latest_price, get_company_financials, get_stock_history, get_technical_indicators],
    middleware=[],
    system_prompt=(
        "You help users research stocks. "
        "For questions about buying or selling now, fetch the latest price, "
        "technical indicators, and company financials before answering. "
        "For Google, use GOOGL unless the user specifies GOOG, "
        "and state that assumption. "
        "Search stock lessons for trading concepts and cite available sources. "
        "Report data timestamps and financial statement periods. "
        "Explain tool failures and missing fields specifically. "
        "Never invent prices, indicators, or future growth forecasts. "
        "Distinguish adjusted historical prices from current quoted prices."
    ),
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "trading-test"}}

print("Sending question...", flush=True)

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "CAN I buy google stock now? "                
            ),
        }]
    },
    config = {"configurable": {"thread_id": "rag-clean-002"}}
)

# Show tool calls, tool results, and the final answer.
for message in result["messages"]:
    message.pretty_print()